<a href="https://colab.research.google.com/github/Safwanach2002/ML/blob/main/Decision_Tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# DECISION TREE CLASSIFICATION

# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Mount Google Drive
drive.mount('/content/drive')

# 1. Load and Understand the Dataset

df = pd.read_excel('/content/drive/MyDrive/Decision_Tree_Dataset.xlsx')

# Display first 10 records
print(df.head(10))

# Identify input features and target
X = df.drop('Result', axis=1)
y = df['Result']

print("Input Features:", X.columns)
print("Target Variable: Result")

# Class Distribution
print(y.value_counts())
sns.countplot(x=y)
plt.title("Class Distribution")
plt.show()


# 2. Entropy, Gini Impurity and Information Gain

# Calculate Entropy and Gini
p = y.value_counts(normalize=True)
entropy = -sum(p * np.log2(p))
gini = 1 - sum(p ** 2)

print("Entropy:", entropy)
print("Gini Impurity:", gini)

# Select a numerical feature
feature = X.select_dtypes(include=np.number).columns[0]
thresholds = sorted(X[feature].unique())[:-1]

# Calculate Information Gain
gains = []

for t in thresholds:
    left = y[X[feature] <= t]
    right = y[X[feature] > t]

    def ent(a):
        p = a.value_counts(normalize=True)
        return -sum(p * np.log2(p))

    gain = entropy - (len(left)/len(y)*ent(left) + len(right)/len(y)*ent(right))
    gains.append(gain)

best = np.argmax(gains)
print("Best Split:", feature, "<=", thresholds[best])
print("Best Information Gain:", gains[best])

plt.plot(thresholds, gains, marker='o')
plt.xlabel("Split Threshold")
plt.ylabel("Information Gain")
plt.title("Information Gain vs Threshold")
plt.show()


# 3. Decision Tree Construction

# Demonstrate root split
t = thresholds[best]
print("Root Node:", feature, "<=", t)
print("Left Branch:", y[X[feature] <= t].value_counts())
print("Right Branch:", y[X[feature] > t].value_counts())

# Train Decision Tree
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42
)

model = DecisionTreeClassifier(criterion="gini", random_state=42)
model.fit(X_train, y_train)

# Visualize Decision Tree
plt.figure(figsize=(15, 8))
plot_tree(model, feature_names=X.columns,
          class_names=[str(c) for c in model.classes_],
          filled=True)
plt.show()


# 4. Prediction and Model Evaluation

y_pred = model.predict(X_test)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# Evaluation Metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, average='weighted', zero_division=0))
print("Recall:", recall_score(y_test, y_pred, average='weighted', zero_division=0))
print("F1 Score:", f1_score(y_test, y_pred, average='weighted', zero_division=0))

# TP, TN, FP and FN are values shown in the confusion matrix.


# 5. Feature Importance and Tree Complexity

# Feature Importance
importance = pd.Series(model.feature_importances_, index=X.columns)
importance.sort_values().plot(kind='barh')
plt.title("Feature Importance")
plt.show()

# Check Different Tree Depths
depths = range(1, 11)
train_acc = []
test_acc = []

for d in depths:
    tree = DecisionTreeClassifier(max_depth=d, random_state=42)
    tree.fit(X_train, y_train)
    train_acc.append(tree.score(X_train, y_train))
    test_acc.append(tree.score(X_test, y_test))

plt.plot(depths, train_acc, marker='o', label='Training')
plt.plot(depths, test_acc, marker='o', label='Testing')
plt.xlabel("Tree Depth")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Tree Depth vs Accuracy")
plt.show()

# Effect of min_samples_split and min_samples_leaf
for n in [2, 5, 10]:
    tree = DecisionTreeClassifier(min_samples_split=n,
                                  min_samples_leaf=2,
                                  random_state=42)
    tree.fit(X_train, y_train)
    print("min_samples_split =", n, "Accuracy =", tree.score(X_test, y_test))


# 6. Gini vs Entropy and New Prediction

# Compare Gini and Entropy
for c in ['gini', 'entropy']:
    tree = DecisionTreeClassifier(criterion=c, random_state=42)
    tree.fit(X_train, y_train)
    print(c, "Testing Accuracy:", tree.score(X_test, y_test))
    print("Tree Depth:", tree.get_depth())
    print("Number of Leaves:", tree.get_n_leaves())

# Create Three New Student Records
new_students = X.iloc[:3].copy()

# Predict Pass or Fail
print("New Student Predictions:", model.predict(new_students))
